# Workshop: Batch & Stream Ingestion

Practice COPY INTO, Auto Loader, schema evolution, and streaming patterns with hands-on exercises.

| Duration | Format | Difficulty |
|---|---|---|
| 30 min | Hands-on Workshop | Intermediate |

**Prerequisites:** 02 — Batch & Stream Ingestion Demo

<!-- TRAINER-BOX -->

> **TRAINER INSTRUCTIONS**
>
> | Tier | Target audience | Time | Goal |
> |---|---|---|---|
> | **PART 1 — FUNDAMENTAL** | New to Spark / Databricks | 70 min | Building confidence, fill-in-blank |
> | **PART 2 — ADVANCED** | 1+ year experience | 65 min | Debugging, optimization, design from scratch |
>
> - **Beginner** groups: complete all PART 1, PART 2 optional
> - **Experienced** groups: PART 1 as quick review (15 min), focus on PART 2
> - **Mixed** groups: PART 1 mandatory, PART 2 for faster participants

<!-- PART1-FUNDAMENTAL -->

---

# PART 1 — FUNDAMENTAL (Fundamental Level)

---

<!-- LAB-SCENARIO -->

## Scenario

> *"New files keep arriving in cloud storage (S3/ADLS). Instead of full re-scans, you need to implement Auto Loader (`cloudFiles`) to process only new files incrementally — with schema inference and checkpointing."*

<!-- LAB-OBJECTIVES -->

## Learning Objectives

After completing this lab you will be able to:

- Configure Auto Loader source using `cloudFiles` format options
- Enable schema inference and automatic schema evolution
- Use `trigger(availableNow=True)` for batch-style incremental runs
- Implement checkpointing for fault tolerance and restartability
- Monitor Auto Loader progress in Spark UI and query metrics

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Prepare landing zone and checkpoint paths
landing_path = f"{DATASET_PATH}/orders/stream"
checkpoint_path = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/checkpoint"
schema_path = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/schema"
target_table = f"{CATALOG}.{BRONZE_SCHEMA}.orders_stream"

# Clean up from previous runs
spark.sql(f"DROP TABLE IF EXISTS {target_table}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

# Ensure customers table exists (needed for Task 8: Stream-Static Join)
customers_path = f"{DATASET_PATH}/customers/customers.csv"
if not spark.catalog.tableExists(f"{CATALOG}.{BRONZE_SCHEMA}.customers"):
    spark.read.format("csv").option("header", True).option("inferSchema", True) \
        .load(customers_path).write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
    print(f"Created: {CATALOG}.{BRONZE_SCHEMA}.customers")

print(f"Landing path: {landing_path}")
print(f"Target table: {target_table}")
print(f"Files available: {[f.name for f in dbutils.fs.ls(landing_path)]}")

---
## Task 1: COPY INTO (Batch Ingestion) ~3 min

Use `COPY INTO` to load JSON files from the landing zone into a Delta table.

**What you need to do:** Fill in the `FILEFORMAT` parameter (hint: `JSON`).

**Guidance — Task 01**

The goal is to load raw JSON files into a Delta table using the simplest batch ingestion command — `COPY INTO`.

**How COPY INTO works**
`COPY INTO` is a SQL command that reads files from a source path and appends them to an existing Delta table. It tracks which files have been loaded using file-level metadata, so running it twice on the same files produces zero new rows. The command requires a `FILEFORMAT` parameter — in this case `JSON`.

**What the setup code does**
The first `CREATE OR REPLACE TABLE` statement defines the target schema. You only need to fill in the file format. The `FORMAT_OPTIONS` with `mergeSchema` allows minor schema differences between files.

**Things to think about**
- Why does `COPY INTO` need the target table to exist before loading?
- How does it differ from a simple `INSERT INTO ... SELECT * FROM read_files(...)`?
- What happens if the source files contain a column not in the table schema?

In [ ]:
# First create the target table
spark.sql(f"""
    CREATE OR REPLACE TABLE {target_table}
    (order_id STRING, customer_id STRING, product_id STRING, store_id STRING,
     order_datetime STRING, quantity BIGINT, unit_price DOUBLE,
     discount_percent BIGINT, total_amount DOUBLE, payment_method STRING)
""")

# TODO: Use COPY INTO to load JSON files
spark.sql(f"""
    COPY INTO {target_table}
    FROM '{landing_path}'
    FILEFORMAT = ________
    FORMAT_OPTIONS ('mergeSchema' = 'true')
""")

count_after_copy = spark.table(target_table).count()
print(f"Rows after COPY INTO: {count_after_copy}")

In [ ]:
# -- Validation --
assert count_after_copy > 0, "COPY INTO should have loaded data"
print(f"Task 1 OK: {count_after_copy} rows loaded via COPY INTO")

---
## Task 2: Verify COPY INTO Idempotency ~2 min

Run COPY INTO again on the **same files**. Observe that 0 new rows are loaded — COPY INTO tracks which files were already processed.

**What you need to do:** Run the cell and verify the row count stays the same.

**Guidance — Task 02**

The goal is to prove that `COPY INTO` is **idempotent** — running it again on the same files loads zero new rows.

**Why idempotency matters**
In production, pipelines can fail mid-run and need to be safely retried. If the ingestion command is not idempotent, retrying would create duplicates. `COPY INTO` avoids this by storing the list of already-loaded file paths as metadata on the target table.

**What to observe**
Run the cell and compare the counts. The difference should be exactly zero. This is the key property that makes `COPY INTO` safe for scheduled batch jobs — even if the scheduler fires twice.

**Things to think about**
- Where does COPY INTO store the "already loaded" file list — in the Delta log or externally?
- If you `DROP` and recreate the table, will COPY INTO re-load the same files?

In [ ]:
# TODO: Re-run the same COPY INTO
spark.sql(f"""
    COPY INTO {target_table}
    FROM '{landing_path}'
    FILEFORMAT = JSON
    FORMAT_OPTIONS ('mergeSchema' = 'true')
""")

count_after_rerun = spark.table(target_table).count()
print(f"Rows after re-run: {count_after_rerun} (was {count_after_copy})")
print(f"New rows loaded: {count_after_rerun - count_after_copy}")

In [ ]:
# -- Validation --
assert count_after_rerun == count_after_copy, "COPY INTO should be idempotent - no new rows!"
print("Task 2 OK: COPY INTO is idempotent. 0 new rows on re-run.")

---
## Task 3: Auto Loader — Configure Stream ~5 min

Set up Auto Loader (`cloudFiles`) to read JSON files from the landing zone.

**What you need to do:** Fill in the blanks:
1. `.format(________)` → `"cloudFiles"`
2. `.option("cloudFiles.format", ________)` → `"json"`

**Guidance — Task 03**

The goal is to configure Auto Loader — Databricks' recommended way to ingest files at scale using Structured Streaming.

**How Auto Loader works**
Auto Loader uses the `cloudFiles` format as its Structured Streaming source. The actual file format (JSON, CSV, Parquet) is specified separately via the `cloudFiles.format` option. This two-level approach allows Auto Loader to handle file discovery and tracking independently of file parsing.

**Schema management**
The `schemaLocation` option tells Auto Loader where to persist the inferred schema. On first run, it scans the files and writes the schema to this path. On subsequent runs, it reuses the cached schema — avoiding expensive re-inference. The `inferColumnTypes` option promotes strings to proper types (INT, DOUBLE, etc.).

**Things to think about**
- Why does Auto Loader need a separate `schemaLocation` path?
- What happens if you delete the schema location and re-run?
- How does `cloudFiles` compare to `COPY INTO` for millions of files?

In [ ]:
# Reset target for Auto Loader test
al_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_autoloader"
spark.sql(f"DROP TABLE IF EXISTS {al_target}")
dbutils.fs.rm(checkpoint_path, True)
dbutils.fs.rm(schema_path, True)

In [ ]:
# TODO: Configure Auto Loader readStream
df_stream = (
    spark.readStream
    .format(________)
    .option("cloudFiles.format", ________)
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
)

In [ ]:
# -- Validation --
assert df_stream.isStreaming, "Should be a streaming DataFrame"
print(f"Task 3 OK: Streaming DataFrame configured with schema: {df_stream.schema.fieldNames()}")

---
## Task 4: Write Stream with trigger(availableNow=True) ~3 min

Write the streaming DataFrame to a Delta table. `availableNow=True` processes all available files and stops.

**What you need to do:** Fill in `.trigger(________=________)` → `.trigger(availableNow=True)`

**Guidance — Task 04**

The goal is to write the streaming DataFrame to a Delta table using the `availableNow` trigger mode.

**Trigger modes explained**
Structured Streaming supports several trigger modes. `availableNow=True` is the most common for batch-like pipelines — it processes all pending files in multiple micro-batches and then stops. This is different from `once=True` (legacy, single micro-batch) and `processingTime` (continuous, runs forever at intervals).

**The writeStream chain**
The writer follows a builder pattern: `.format("delta")` → `.outputMode("append")` → `.option("checkpointLocation", ...)` → `.trigger(...)` → `.toTable(...)`. The checkpoint is essential — it stores which files and offsets have been processed. Without it, every run would reprocess everything.

**Things to think about**
- When would you use `processingTime="10 seconds"` instead of `availableNow=True`?
- What is stored in the checkpoint directory — and what happens if you delete it?

In [ ]:
# TODO: Write stream to Delta table
query = (
    df_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(________=________)
    .toTable(al_target)
)

query.awaitTermination()
print(f"Stream completed. Rows loaded: {spark.table(al_target).count()}")

In [ ]:
# -- Validation --
al_count = spark.table(al_target).count()
assert al_count > 0, "Auto Loader should have loaded data"
print(f"Task 4 OK: {al_count} rows loaded via Auto Loader")

---
## Task 5: Incremental Processing ~3 min

Re-run the stream with the **same checkpoint**. Since no new files arrived, 0 new rows should be processed — this proves checkpoint-based tracking works.

**What you need to do:** Run the cell and verify incremental behavior (0 new rows).

**Guidance — Task 05**

The goal is to verify that checkpoint-based tracking works — re-running the stream with the same checkpoint processes zero new rows.

**How checkpoints enable incremental processing**
When Auto Loader runs with a checkpoint, it records the offsets (which files it has seen) and the committed micro-batch IDs. On the next run, it only looks for files newer than what was previously recorded. This is what makes streaming pipelines truly incremental — even across cluster restarts.

**Things to think about**
- What is the difference between COPY INTO's file tracking and Auto Loader's checkpoint tracking?
- In production, where should you store checkpoints — local disk, DBFS, or cloud storage?

In [ ]:
# Re-run the stream (same checkpoint = incremental)
df_stream2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
)

query2 = (
    df_stream2
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(al_target)
)

query2.awaitTermination()
al_count2 = spark.table(al_target).count()
print(f"Rows after re-run: {al_count2} (was {al_count})")
print(f"New rows: {al_count2 - al_count}")

In [ ]:
# -- Validation --
assert al_count2 == al_count, f"Should be 0 new rows, but got {al_count2 - al_count}"
print("Task 5 OK: Incremental processing verified. 0 new rows on re-run (checkpoint works!)")

---
## Task 6: Metadata Columns ~3 min

Add metadata columns to streaming data for **debugging and auditing** in production.

**What you need to do:**
1. Configure Auto Loader readStream (same as Task 3)
2. Add `.withColumn("_processing_time", ...)` using `current_timestamp()`
3. Add `.withColumn("_source_file", ...)` using `col("_metadata.file_path")`
4. Write the enriched stream to a Delta table

**Guidance — Task 06**

The goal is to enrich streaming data with **metadata columns** that are essential for debugging and auditing in production.

**Why metadata matters**
In a multi-source pipeline, you often need to know *when* a row was processed and *which file* it came from. `current_timestamp()` captures the processing time, and `_metadata.file_path` (a hidden Spark column available in file-based sources) gives the source file path. These columns make troubleshooting much easier when data issues arise.

**The `_metadata` struct**
Spark automatically provides a `_metadata` struct column for file-based data sources. It contains fields like `file_path`, `file_name`, `file_size`, and `file_modification_time`. You access them via `col("_metadata.file_path")`.

**Things to think about**
- Why use `current_timestamp()` instead of `_metadata.file_modification_time`?
- Would `_metadata` work the same way when reading from a Delta table (not files)?

In [ ]:
from pyspark.sql.functions import current_timestamp, col

# Paths and reset
metadata_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata"
metadata_checkpoint = f"/tmp/{CATALOG}/lab05/checkpoint_metadata"
metadata_schema = f"/tmp/{CATALOG}/lab05/schema_metadata"

spark.sql(f"DROP TABLE IF EXISTS {metadata_target}")
dbutils.fs.rm(metadata_checkpoint, True)
dbutils.fs.rm(metadata_schema, True)

In [ ]:
# TODO: Configure Auto Loader with metadata columns
df_with_metadata = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", metadata_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
    .withColumn("_processing_time", ________)       # hint: current_timestamp()
    .withColumn("_source_file", ________)            # hint: col("_metadata.file_path")
)

In [ ]:
# Write enriched stream to Delta
(
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", metadata_checkpoint)
    .trigger(availableNow=True)
    .toTable(metadata_target)
    .awaitTermination()
)

print(f"Written to: {metadata_target}")
display(spark.table(metadata_target).limit(5))

In [ ]:
# -- Validation --
meta_cols = spark.table(metadata_target).columns
assert "_processing_time" in meta_cols, "Missing '_processing_time' — did you use current_timestamp()?"
assert "_source_file" in meta_cols, "Missing '_source_file' — did you use _metadata.file_path?"
print(f"Task 6 OK: Metadata columns added — {meta_cols}")

---
## Task 7: Schema Evolution — Rescued Data ~4 min

Configure Auto Loader with a **partial schema** (only 2 columns). Extra columns from the source files will be captured in `_rescued_data`.

**What you need to do:**
1. Define a partial schema with only `order_id` and `customer_id`
2. Set `cloudFiles.schemaEvolutionMode` to `"rescue"`
3. Pass the partial schema using `.schema(partial_schema)`
4. Verify that `_rescued_data` column contains the extra fields

**Guidance — Task 07**

The goal is to demonstrate Auto Loader's **rescue mode** — handling schema mismatches without losing data.

**The schema evolution problem**
In production, source files evolve — new columns appear, types change. If you define a partial schema (only 2 columns), what happens to the other 8+ columns? With `schemaEvolutionMode = "rescue"`, Auto Loader captures them in a special `_rescued_data` column as a JSON string. No data is lost.

**Three schema evolution modes**
- `addNewColumns` — merges new columns into the schema automatically
- `rescue` — extra columns go into `_rescued_data`, known columns parsed normally
- `failOnNewColumns` — stream fails if unknown columns appear (strictest)

**Using `.schema()` with Auto Loader**
When you pass a partial schema via `.schema(partial_schema)`, Auto Loader applies it *instead of* inferring. Combined with rescue mode, this gives you a stable schema while preserving unexpected data.

**Things to think about**
- When would you prefer `addNewColumns` over `rescue`?
- How would you extract specific fields from the `_rescued_data` JSON string?

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

rescue_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued"
rescue_checkpoint = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/checkpoint_rescue"
rescue_schema = f"{DATASET_PATH}/tmp/{CATALOG}/lab05/schema_rescue"

spark.sql(f"DROP TABLE IF EXISTS {rescue_target}")
dbutils.fs.rm(rescue_checkpoint, True)
dbutils.fs.rm(rescue_schema, True)

# Partial schema — only 2 columns out of many
partial_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
])

In [ ]:
# TODO: Configure Auto Loader with rescue mode
df_rescued = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", rescue_schema)
    .option("cloudFiles.schemaEvolutionMode", "________")  # hint: "rescue"
    .schema(partial_schema)
    .load(landing_path)
)

In [ ]:
# Write with rescued data
(
    df_rescued
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", rescue_checkpoint)
    .trigger(availableNow=True)
    .toTable(rescue_target)
    .awaitTermination()
)

print(f"Written to: {rescue_target}")
display(spark.table(rescue_target).limit(5))

In [ ]:
# -- Validation --
rescue_cols = spark.table(rescue_target).columns
assert "_rescued_data" in rescue_cols, "Missing '_rescued_data' column — did you set schemaEvolutionMode to 'rescue'?"
rescue_count = spark.table(rescue_target).filter("_rescued_data IS NOT NULL").count()
assert rescue_count > 0, "Expected rescued data for columns not in partial schema"
print(f"Task 7 OK: {rescue_count} rows with rescued data (extra columns captured in _rescued_data)")

---
## Task 8: Stream-Static Join ~5 min

Join **streaming orders** with a **static customers** table to enrich the stream. The static side is re-read on every micro-batch.

**What you need to do:**
1. Read `customers` table as a static DataFrame
2. Read orders table as a stream (`.readStream.format("delta")`)
3. Join them on `customer_id` with a `left` join
4. Write the enriched stream to a Delta table

**Guidance — Task 08**

The goal is to **join streaming data with a static table** — one of the most common patterns in production pipelines.

**Stream-static join pattern**
In a stream-static join, one side is a streaming DataFrame (orders) and the other is a regular DataFrame (customers). Spark re-reads the static side on every micro-batch, so if the customer table gets updated between batches, the stream picks up the latest values automatically.

**Join mechanics**
Use `.join(static_df, on="column_name", how="left")`. A `left` join ensures all orders are preserved even if a customer is missing. The result is a streaming DataFrame that you write to Delta as usual.

**Things to think about**
- Can you join two streaming DataFrames together? What restrictions apply?
- Why is `left` join safer than `inner` join for this use case?
- What would happen if the customers table had duplicate `customer_id` values?

In [ ]:
join_target = f"{CATALOG}.{BRONZE_SCHEMA}.orders_enriched"
join_checkpoint = f"/tmp/{CATALOG}/lab05/checkpoint_enriched"

spark.sql(f"DROP TABLE IF EXISTS {join_target}")
dbutils.fs.rm(join_checkpoint, True)

# Static side: read customers table
customers_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")

In [ ]:
# TODO: Read orders as a stream and join with customers
orders_stream = (
    spark.readStream
    .format("________")        # hint: "delta"
    .table(________)           # hint: target_table
)

enriched_stream = orders_stream.join(
    ________,                  # hint: customers_df
    on="________",             # hint: "customer_id"
    how="left"
)

In [ ]:
# TODO: Write the joined stream
(
    enriched_stream
    .writeStream
    .format("delta")
    .outputMode("________")          # hint: "append"
    .option("checkpointLocation", join_checkpoint)
    .trigger(availableNow=________)  # hint: True
    .toTable(________)               # hint: join_target
    .awaitTermination()
)

print(f"Stream-static join written to: {join_target}")
display(spark.table(join_target).limit(5))

In [ ]:
# -- Validation --
enriched_count = spark.table(join_target).count()
enriched_cols = spark.table(join_target).columns
assert enriched_count > 0, "No rows in enriched table"
assert "customer_id" in enriched_cols, "Missing customer_id"
assert any(c for c in enriched_cols if c not in spark.table(target_table).columns), \
    "Join didn't add any new columns — check join expression"
print(f"Task 8 OK: {enriched_count} enriched rows. Columns: {enriched_cols}")

---
## Task 9: Change Data Feed (CDF) for Incremental ETL ~5 min

Use **Change Data Feed** to read only row-level changes from a Delta table.

**What you need to do:**
1. Create a table with `TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')`
2. Insert initial data, then make changes (UPDATE + INSERT)
3. Read changes using `table_changes()` SQL function
4. Filter to `insert` and `update_postimage` for incremental ETL

> **Key columns:** `_change_type` (`insert`, `update_preimage`, `update_postimage`, `delete`), `_commit_version`, `_commit_timestamp`

**Guidance — Task 09**

The goal is to use **Change Data Feed (CDF)** to capture row-level changes and build an incremental ETL pipeline.

**What CDF provides**
When `delta.enableChangeDataFeed` is enabled on a table, every INSERT, UPDATE, and DELETE is recorded with metadata columns: `_change_type` (values: `insert`, `update_preimage`, `update_postimage`, `delete`), `_commit_version`, and `_commit_timestamp`. You read these changes using the `table_changes()` SQL function.

**The incremental ETL pattern**
For a downstream table that should reflect the "latest state", you typically filter for `_change_type IN ('insert', 'update_postimage')` — these represent new rows and the *new* values of updated rows. The `update_preimage` (old values) and `delete` records are useful for auditing but usually excluded from the target.

**Things to think about**
- How does CDF differ from `DESCRIBE HISTORY` — both show changes, right?
- When would you use CDF over Auto Loader for incremental processing?
- What is the storage overhead of enabling CDF on a high-volume table?

In [ ]:
# Setup: create CDF-enabled table
cdf_source = f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_lab"
cdf_target = f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_incremental"

spark.sql(f"DROP TABLE IF EXISTS {cdf_source}")
spark.sql(f"DROP TABLE IF EXISTS {cdf_target}")

In [ ]:
# TODO: Create table with Change Data Feed enabled
spark.sql(f"""
    CREATE TABLE {cdf_source} (
        order_id INT, customer_id INT, amount DOUBLE, status STRING
    )
    TBLPROPERTIES (________ = ________)
""")
# hint: 'delta.enableChangeDataFeed' = 'true'

# Insert initial data
spark.sql(f"""
    INSERT INTO {cdf_source} VALUES
    (1, 101, 99.99, 'pending'),
    (2, 102, 149.50, 'pending'),
    (3, 103, 75.00, 'pending'),
    (4, 101, 200.00, 'pending'),
    (5, 104, 50.00, 'pending')
""")

initial_version = spark.sql(f"DESCRIBE HISTORY {cdf_source}").first()["version"]
print(f"CDF table created at version {initial_version} with {spark.table(cdf_source).count()} rows")

In [ ]:
# Step 2: Make some changes (simulate business operations)
# Update: mark orders 1,2 as shipped
spark.sql(f"UPDATE {cdf_source} SET status = 'shipped' WHERE order_id IN (1, 2)")

# Insert: new order
spark.sql(f"INSERT INTO {cdf_source} VALUES (6, 105, 320.00, 'pending')")

print(f"Changes applied: 2 updates + 1 insert")
print(f"Current table has {spark.table(cdf_source).count()} rows")

In [ ]:
# TODO: Read the Change Data Feed starting from the version AFTER initial load
# Use table_changes() SQL function

df_changes = spark.sql(f"""
    SELECT * FROM ________('{cdf_source}', {initial_version + 1})
    ORDER BY _commit_version, order_id
""")
# hint: table_changes(...)

display(df_changes)

In [ ]:
# TODO: Build incremental ETL — read only inserts and update_postimage (new values)
# Filter out update_preimage and delete to get "current state of changes"

df_incremental = df_changes.filter(
    col("_change_type").isin(________, ________)
)
# hint: "insert", "update_postimage"

# Write incremental changes to target
df_incremental.drop("_change_type", "_commit_version", "_commit_timestamp").write \
    .mode("append").saveAsTable(cdf_target)

print(f"Incremental ETL: {df_incremental.count()} rows written to {cdf_target}")
display(spark.table(cdf_target))

In [ ]:
# -- Validation --
cdf_count = spark.table(cdf_target).count()
assert cdf_count == 3, f"Expected 3 rows (2 updated + 1 inserted), got {cdf_count}"
statuses = [r["status"] for r in spark.table(cdf_target).collect()]
assert "shipped" in statuses, "Should contain 'shipped' status from updates"
assert "pending" in statuses, "Should contain 'pending' status from new insert"
print(f"Task 9 OK: CDF incremental ETL — {cdf_count} change rows captured")
print("  2 update_postimage (shipped) + 1 insert (new order)")

<!-- PART2-ADVANCED -->

---

# PART 2 — ADVANCED (Bonus — If Time Permits)

> **For whom:** Participants with 1+ year of Spark/Databricks experience
>
> **Rules:**
> - No scaffold `= None` — write from scratch
> - Tasks described as JIRA tickets (requirements + acceptance criteria)
> - Complete **at least 2 of 4 challenges**
> - Time per challenge: 8-15 minutes

---

### Bug Hunt — Auto Loader Does Not Process New Files

After cluster restart, Auto Loader starts processing **from the beginning** (duplicates!).

```python
df_stream = (spark.readStream
             .format('cloudFiles')
             .option('cloudFiles.format', 'json')
             .option('cloudFiles.schemaLocation', SCHEMA_PATH)
             .load(LANDING_PATH))

query = (df_stream.writeStream
         .format('delta')
         .outputMode('append')
         .trigger(availableNow=True)
         .start(BRONZE_TABLE_PATH))  # bug: missing checkpointLocation
```

Find the bug, fix it, explain what the checkpoint stores.

**Acceptance criteria:**
- Identified bug with explanation
- Fixed code with `checkpointLocation`
- Explanation: what checkpoint stores (offset, schema)
- Demonstration: run 2x — second time 0 new rows

In [ ]:
# YOUR SOLUTION — Auto Loader Does Not Process New Files
# ------------------------------------------------------------



### Design From Scratch — Multi-Format Auto Loader

The `landing/` folder receives files from 3 systems simultaneously:
- `customers_*.csv` (separator `;`, header=true)
- `orders_*.json`
- `products_*.parquet`

Design a pipeline: one stream or multiple? Justify your choice.
Implement your chosen architecture.

**Acceptance criteria:**
- Justified decision: 1 stream with `pathGlobFilter` or 3 separate streams
- Each format goes to a separate Bronze table
- Each stream has its own `checkpointLocation`
- Column `_metadata.file_path` added to each table

In [ ]:
# YOUR SOLUTION — Multi-Format Auto Loader
# ------------------------------------------------------------



### Performance — availableNow vs processingTime — When to Use What?

You have 2 pipelines:
- **Pipeline A**: critical — new data must be visible within 30 seconds
- **Pipeline B**: batch overnight — processes 10 GB total per night

Choose the trigger mode for each, measure actual behavior on test data.

**Acceptance criteria:**
- Correct trigger for Pipeline A with justification
- Correct trigger for Pipeline B with justification
- Demonstrate `query.lastProgress['numInputRows']` and `'processedRowsPerSecond'`
- Written answer: when NOT to use `continuous` trigger

In [ ]:
# YOUR SOLUTION — availableNow vs processingTime — When to Use What?
# ------------------------------------------------------------



### Multi-Step Pipeline — Bronze → Silver → Gold Increment in 3 Steps

Build a complete incremental pipeline from scratch:

**Step 1 — Bronze:** Auto Loader `orders/*.json` → `bronze_orders_stream` (append)
**Step 2 — Silver:** `foreachBatch` MERGE INTO `silver_orders` (upsert by `order_id`)
**Step 3 — Gold:** CDF from `silver_orders` → `gold_daily_revenue` (sum amount GROUP BY date)

Each step is a separate stream with its own checkpoint.

**Acceptance criteria:**
- All 3 tables exist with data
- Silver is idempotent (re-run = no duplicates)
- Gold updates incrementally (CDF with `startingVersion`)
- `query.lastProgress` available for each stream

In [ ]:
# YOUR SOLUTION — Bronze → Silver → Gold Increment in 3 Steps
# ------------------------------------------------------------



---

## Part 2 Summary

You have completed the Advanced section for **Batch & Stream Ingestion**.

**Reflection (optional):**
- Which challenge was the hardest and why?
- What would you change in your implementation?
- What approach would you use in production?

---

---
## Cleanup

In [ ]:
# Stop any active streams
for s in spark.streams.active:
    s.stop()

# Drop lab tables
for t in [target_table, al_target, 
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_with_metadata",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_rescued",
          f"{CATALOG}.{BRONZE_SCHEMA}.orders_enriched",
          f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_lab",
          f"{CATALOG}.{BRONZE_SCHEMA}.cdf_orders_incremental"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Clean temp paths
for p in [checkpoint_path, schema_path,
          f"/tmp/{CATALOG}/lab05/checkpoint_metadata",
          f"/tmp/{CATALOG}/lab05/schema_metadata",
          f"/tmp/{CATALOG}/lab05/checkpoint_rescue",
          f"/tmp/{CATALOG}/lab05/schema_rescue",
          f"/tmp/{CATALOG}/lab05/checkpoint_enriched"]:
    dbutils.fs.rm(p, True)

print("Lab cleanup complete")

---
## Lab Complete!

You have:
- Used COPY INTO for idempotent batch loading
- Configured Auto Loader (cloudFiles) for streaming ingestion
- Used trigger(availableNow=True) for incremental processing
- Verified checkpoint-based exactly-once guarantees
- Added metadata columns (`_processing_time`, `_source_file`) to streaming writes
- Used rescued data column for schema evolution handling
- Performed a stream-static join to enrich streaming data
- Used Change Data Feed (CDF) for incremental ETL

| Feature | COPY INTO | Auto Loader | CDF |
|---------|-----------|-------------|-----|
| Format | SQL command | readStream/writeStream | readChangeFeed / table_changes() |
| Scalability | Thousands of files | Millions of files | Any Delta table |
| Schema evolution | Manual | Automatic (rescue column) | Follows source schema |
| File tracking | SQL state | Checkpoint directory | Version-based |
| Use case | Simple batch | File-based streaming | Change-based incremental |

> **Pro Tip:** Auto Loader uses `cloudFiles` format. COPY INTO is simpler but Auto Loader scales better (file notification mode for millions of files). CDF captures row-level changes and is ideal for propagating updates through Medallion layers.

> **Next:** [02 -- Lakeflow Pipelines Workshop](02_lakeflow_pipelines_workshop.ipynb)

---

← [02 — Batch & Stream Ingestion](../Demo/02_batch_stream_ingestion_demo.ipynb) | **[ README](../../../README.md)** | [03 — Lakeflow Pipelines](../Demo/03_lakeflow_pipelines_demo.ipynb) →
